# Aksara OCR — ConvNeXt-Tiny fix

ConvNeXt-Tiny diverged in the backbone run (macro-F1 0.00) because lr=1e-3 is
too high for it. This reruns *only* ConvNeXt at lr=1e-4.

**Setup:** T4 x2, Internet On, **Save & Run All (Commit)**. ~3 runs x ~40 min
≈ 2 h — fits one session.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Settings (right sidebar) > Accelerator > GPU T4 x2, then rerun."
)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
supported = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]

print(f"{name}  ({arch})")
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
print(f"torch {torch.__version__}  supports: {supported}")

# torch.cuda.is_available() returns True even when this build ships no kernels
# for the device - the failure then surfaces as a warning storm with every run
# landing in failures.jsonl. Check the architecture explicitly and stop here.
if arch not in supported:
    raise SystemExit(
        f"{name} is {arch}, but this PyTorch build only has kernels for "
        f"{supported}. Switch Settings > Accelerator to GPU T4 x2 (sm_75) "
        f"and rerun. The P100 is sm_60 and will not work."
    )

# Prove a real kernel runs, not just that a device is listed.
probe = (torch.randn(512, 512, device="cuda") @ torch.randn(512, 512, device="cuda")).sum()
torch.cuda.synchronize()
print(f"GPU compute OK (probe={probe.item():.1f})")

In [ ]:
# Internet must be ON (Settings > Internet) for these three lines.
import os
from pathlib import Path

REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"
REPO = Path("/kaggle/working/aksantara-ocr")

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Kaggle; installing the rest avoids a slow reinstall
# of torch against a possibly-mismatched CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"ready: {Path.cwd()}")

In [ ]:
# The diverged ConvNeXt runs wrote result.json files, and the runner skips any
# run whose result.json exists. Delete them so the fixed config actually reruns.
import shutil
from pathlib import Path
ARTIFACTS = Path("/kaggle/working/artifacts")
RESULTS = ARTIFACTS / "results" / "backbones"
RESULTS.mkdir(parents=True, exist_ok=True)

# Restore other finished backbones (so they are not redone) but NOT convnext.
sources  = list(REPO.glob("artifacts_kaggle/**/results/backbones"))
sources += list(Path("/kaggle/input").glob("*/artifacts/results/*"))
restored = purged = 0
for cand in sources:
    if not cand.is_dir(): continue
    for run in cand.iterdir():
        if not (run/"result.json").exists(): continue
        if "convnext" in run.name:   # skip the broken ones on purpose
            continue
        dest = RESULTS/run.name
        if not dest.exists() and "__unified__" in run.name:
            shutil.copytree(run, dest); restored += 1
# also purge any convnext already sitting in the working dir
for run in RESULTS.glob("convnext_tiny__*"):
    shutil.rmtree(run); purged += 1
print(f"restored {restored} non-convnext runs, purged {purged} broken convnext runs")

In [ ]:
# Fetch the cleaned, published v3. ~808 MB; needs Internet ON.
DOI = "10.17632/vfj32bpjsf.3"
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /kaggle/working/raw

RAW_ROOT = "/kaggle/working/raw" 

In [ ]:
# Pre-resize once to 224px. Source images run up to 1500x1500; decoding one
# costs ~6.7 ms/core, so without this the dataloader, not the GPU, is the limit.
# A 224px cache drops that to ~1 ms/img.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /kaggle/working/data --size 224

DATA_ROOT = "/kaggle/working/data"
# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}

In [ ]:
# stratified (no writer ids); --drop-duplicates removes images that became
# byte-identical after the 224px resize.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" --out-dir "{ARTIFACTS}" \
    --split-strategy stratified --drop-duplicates

# Verify against published v3. Image/class/script counts come from manifest.csv,
# written before any filtering, so they are invariant.
import pandas as pd
manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {"images": len(manifest), "classes": manifest["label"].nunique(),
          "scripts": manifest["script"].nunique()}
for k, want in EXPECTED.items():
    print(f"  {k:8} {actual[k]:6}  expected {want:6}  {'OK' if actual[k]==want else 'MISMATCH'}")
if actual != EXPECTED:
    raise SystemExit("Data does not match published v3 — re-run fetch and cache cells.")
print("Matches published v3.")

In [ ]:
# Rerun ConvNeXt-Tiny at lr=1e-4.
!python scripts/02_run_matrix.py --config configs/backbone_convnext_fix.yaml     --artifacts "{ARTIFACTS}" --results "{RESULTS}" --num-workers 4 --time-budget 8

In [ ]:
import json
from pathlib import Path
for f in sorted(Path(RESULTS).glob("convnext_tiny__*/result.json")):
    d=json.load(open(f)); m=d["test_metrics"]
    print(f"{f.parent.name}: acc={m['accuracy']*100:.2f}  macro_f1={m['macro_f1']*100:.2f}")

If macro-F1 is now ~98-99, ConvNeXt is fixed. Save Version to keep the output.